In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('../../..'))
from lstm import LSTM
from graph2vecdataset import TimeSeriesDataset
from model_utils.utils import generate_exogenous_features
import time
import itertools
import math
import pickle

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True


In [2]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise
'''
# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")
                    '''

'\n# Clean up directories from previous runs\ndirs_to_cleanup = [\'best_models\', \'grid_search_plots\', \'training_logs\', \'inference_logs\']\nfor dir_path in dirs_to_cleanup:\n    if os.path.exists(dir_path):\n        # Retry a few times in case of transient locks\n        for i in range(3):\n            try:\n                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)\n                print(f"Removed directory: {dir_path}")\n                break\n            except Exception as e:\n                if i < 2:\n                    time.sleep(1) # Wait a bit before retrying\n                else:\n                    print(f"Error removing {dir_path}: {e}")\n                    '

In [3]:
DATA_PATH = '../../../dataset/data_andre.feather'  # Adjusted path
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

#NUM_ITEMS = 100
#df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../../../dataset/data_andre.feather...


In [4]:
import holidays

# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------


# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------


# ensure day 2022-09-24 is the first day of test set
df = df.sort_values([DATE_COL, 'item_id', 'store_id']).reset_index(drop=True)

# -----------------------------------------------------------------------------
# BASIC CALENDAR PARTS
# -----------------------------------------------------------------------------
EXOG_COLS = [
    # base
    "day_of_week", "day_of_month", "week_of_year", "week_of_month",
    "month", "quarter", "is_weekend",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end",

    # special days of the week
    "is_monday", "is_friday",
    # holidays
    "is_holiday", "is_thanksgiving", "is_black_friday",
    "is_christmas", "is_christmas_eve", "is_new_year_eve",
    "is_pre_holiday_1", "is_pre_holiday_2", "is_pre_holiday_3", "is_pre_holiday_7",
    "is_post_holiday_1", "is_post_holiday_2", "is_post_holiday_3", "is_post_holiday_7",

    # boundary / behavior
    "is_bridge_day",
    # promotions
    # "promo_type_FRPG", "promo_value_FRPG",
    # "promo_type_GAS", "promo_value_GAS",
    # "promo_type_BOGO", "promo_value_BOGO",
    # "promo_type_DISC", "promo_value_DISC",
    # "promo_type_CIRC", "promo_value_CIRC",
    # "promo_type_CIRE", "promo_value_CIRE",
    # "promo_type_CLCP", "promo_value_CLCP",
    # "promo_type_LFPE", "promo_value_LFPE"
]
df = generate_exogenous_features(df, date_col=DATE_COL, exog_cols=EXOG_COLS)


1082371


In [5]:
df

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_pre_holiday_2,is_pre_holiday_3,is_pre_holiday_7,is_post_holiday_1,is_post_holiday_2,is_post_holiday_3,is_post_holiday_7,is_bridge_day
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021-01-23,55,6,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021-01-23,71,10,refrigerated baked gds,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021-01-23,116,16,dairy cream,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
4,2021-01-23,128,14,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-22,983332,11,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082367,2023-02-22,983754,4,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082368,2023-02-22,988016,1,sparkling seltzer mixer,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082369,2023-02-22,991921,8,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
product_id = 907969
products= df[df['item_id'] == product_id][['item_id', 'store_id']].drop_duplicates().values
results = []
os.makedirs('grid_search_plots', exist_ok=True)

In [7]:
# Filter dataframe to ONLY the specific product and store!
df = df[(df['item_id'] == products[0][0]) & (df['store_id'] == products[0][1])].sort_values(DATE_COL).reset_index(drop=True)

forecast_horizon = 152
seq_length = 30
train_size = 455
val_size = 154
lookback_window = 7 
BATCH_SIZE = 32
test_start_idx = len(df) - forecast_horizon
val_start_idx = test_start_idx - val_size
train_start_idx = val_start_idx - train_size
train_slice = slice(train_start_idx, val_start_idx)
val_slice = slice(val_start_idx, test_start_idx)
test_slice = slice(test_start_idx, None)
    
print(f"Train slice: {train_slice}, Val slice: {val_slice}, Test slice: {test_slice}")
# Extract Target
train = df[TARGET_COL][train_slice].values
val = df[TARGET_COL][val_slice].values
test = df[TARGET_COL][test_slice].values
# Scale Target
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()
test_scaled = scaler.transform(test.reshape(-1, 1)).flatten()

Train slice: slice(0, 455, None), Val slice: slice(455, 609, None), Test slice: slice(609, None, None)


In [ ]:
import time
from GNN.DynamicSimilarities.Graph2vec.generate_graph2vecwithadaptativethreshold import load_or_generate_embeddings

USE_EMBEDDINGS = True
USE_RESIDUALS = False
MODEL_TYPE = 'ridge'

enable_edges_within_star= True
metric='cid'
window_size=15
step_size=1
threshold=None
percentile=0.5

if USE_EMBEDDINGS:
    start_time = time.time()
    graph_embeddings, graph2vec_model, csv_path = load_or_generate_embeddings(
        product_id=product_id,
        metric=metric,
        window_size=window_size,
        step_size=step_size,
        threshold=threshold,
        enable_edges_within_star=enable_edges_within_star,
        percentile=percentile,
        use_residuals=USE_RESIDUALS,
        model_type=MODEL_TYPE
    )
    end_time = time.time()
    print(f"Time taken for graph & embedding logic: {end_time - start_time:.4f} seconds")
else:
    graph_embeddings = None
    graph2vec_model = None

c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading graphs from c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\GNN\DynamicSimilarities\Graph2vec\..\GraphAnalysis\DynamicGraphPkls\cid\15\1\907969\dynamic_graphs_cid_Window15_Step1_pct0.5.pkl...
Successfully loaded 595 graphs.
Generating Graph2Vec embeddings for 595 valid graphs out of 595...
A extrair subestruturas via WL e a isolar grafos únicos para treino...
A construir vocabulário para 358 assinaturas únicas de grafo...
A treinar em 358 grafos (ignorando redundâncias topológicas)...
Finished generating embeddings!
Total time taken: 0.40 seconds
Saving embeddings to c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\GNN\DynamicSimilarities\Graph2vec\..\GraphAnalysis\DynamicGraphPkls\cid\15\1\907969\embeddings_cid_Window15_Step1_pct0.5.pkl...
Saving model to c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\GNN\DynamicSimilarities\Graph2vec\..\GraphAnalysis\DynamicGraphPkls\cid\15\1\907969\graph2vec_model_cid

In [9]:
# Extract Exogenous Variables
exog_train = None
exog_val = None
exog_cols = EXOG_COLS
full_exog = None

if exog_cols and len(exog_cols) > 0:
    exog_train = df[exog_cols][train_slice].values
    exog_val = df[exog_cols][val_slice].values
    exog_test = df[exog_cols][test_slice].values
        # Scale Exogenous Variables
    exog_scaler = MinMaxScaler()
    exog_train_scaled = exog_scaler.fit_transform(exog_train)
    exog_val_scaled = exog_scaler.transform(exog_val)
    exog_test_scaled = exog_scaler.transform(exog_test)
else:
    exog_train_scaled = None
    exog_val_scaled = None
    exog_test_scaled = None
    # Input size = 1 (target) + number of exog features


In [10]:
if USE_EMBEDDINGS and graph_embeddings is not None:
    embedding_dim = graph_embeddings.shape[1] if len(graph_embeddings.shape) > 1 else 1
else:
    embedding_dim = 0
input_size = 1 + (len(exog_cols) if exog_cols and len(exog_cols) > 0 else 0) +embedding_dim

In [11]:
if USE_EMBEDDINGS and graph_embeddings is not None:
    # Align and pad graph embeddings to match timestamps for train and val sets
    padding = np.zeros((window_size - 1, embedding_dim))
    aligned_embeddings = np.vstack([padding, graph_embeddings])

    emb_train = aligned_embeddings[train_slice]
    emb_val = aligned_embeddings[val_slice]
else:
    aligned_embeddings = None
    emb_train = None
    emb_val = None

#-------------------------------------------------------------------------
# 2. Datasets & Loaders
#-------------------------------------------------------------------------
# Pass exogenous data and embeddings separately to TimeSeriesDataset
train_dataset = TimeSeriesDataset(
    target_data=train_scaled, 
    exog_data=exog_train_scaled if exog_cols and len(exog_cols) > 0 else None, 
    seq_length=seq_length,
    embeddings=emb_train,
    graph_window_size=window_size
)
use_pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=use_pin_memory)
    
val_dataset = TimeSeriesDataset(
    target_data=val_scaled, 
    exog_data=exog_val_scaled if exog_cols and len(exog_cols) > 0 else None, 
    seq_length=seq_length,
    embeddings=emb_val,
    graph_window_size=window_size
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=use_pin_memory)

In [12]:
HIDDEN_SIZE = 32
NUM_LAYERS = 1
DROPOUT = 0.0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTM(input_size=input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)


In [13]:
LEARNING_RATE = 0.001
PATIENCE = 100
criterion = nn.MSELoss()
criterion2 = nn.MSELoss()  # Placeholder for potential multi-task loss

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=PATIENCE//3)

In [ ]:
import time
import itertools
import sys
import os
import importlib
import pickle
from train import train_model
from graph2vecinference_adaptativethreshold import graph2vec_inference
# from utils import compute_metrics # Un-comment when available
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import model_utils.plots
importlib.reload(model_utils.plots)
from model_utils.plots import plot_results

# Grid Search Parameters
seed = 2024
loss_functions = ['MSELoss']  
EPOCHS = 1000

results = []
os.makedirs('grid_search_plots', exist_ok=True)

model_dir_label = f"pct{percentile}" if percentile is not None else "no_emb"
best_models_dir = os.path.join('best_models', f'seed_{seed}', str(window_size), str(step_size), model_dir_label)
os.makedirs(best_models_dir, exist_ok=True)

if USE_EMBEDDINGS:
    prefix_star = "" if enable_edges_within_star else "star_"
    prefix = f"best_lstm_{prefix_star}{product_id}_{metric}_res_{MODEL_TYPE}" if USE_RESIDUALS else f"best_lstm_{prefix_star}{product_id}_{metric}"
    if threshold is not None:
        best_model_path = os.path.join(best_models_dir, f'{prefix}_th{threshold}.pth')
        history_path = os.path.join(best_models_dir, f'{prefix}_th{threshold}_history.pkl')
    if percentile is not None:
        best_model_path = os.path.join(best_models_dir, f'{prefix}_pct{percentile}.pth')
        history_path = os.path.join(best_models_dir, f'{prefix}_pct{percentile}_history.pkl')
else:
    best_model_path = os.path.join(best_models_dir, f'best_lstm_{product_id}_no_emb.pth')
    history_path = os.path.join(best_models_dir, f'best_lstm_{product_id}_no_emb_history.pkl')

if os.path.exists(best_model_path) and os.path.exists(history_path):
    print(f"Loading existing model from {best_model_path} and history from {history_path}")
    model.load_state_dict(torch.load(best_model_path))
    
    with open(history_path, 'rb') as f:
        history = pickle.load(f)
        train_losses = history['train_losses']
        val_losses = history['val_losses']
        best_epoch = history['best_epoch']
        train_time = history['train_time']
else:
    print("Training model...")
    model, train_losses, val_losses, best_epoch, train_time = train_model(
        seed=seed, 
        epochs=EPOCHS, 
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        exog_cols=exog_cols, 
        criterion=criterion, 
        criterion2=criterion2, 
        optimizer=optimizer, 
        device=device, 
        best_model_path=best_model_path, 
        scheduler=scheduler, 
        patience=PATIENCE
    )
    
    # Save the losses and metadata so we don't lose the plot data
    with open(history_path, 'wb') as f:
        pickle.dump({
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_epoch': best_epoch,
            'train_time': train_time
        }, f)


Training model...
Epoch 10/1000 | Train Loss: 0.015708 | Val Loss: 0.008158
Epoch 20/1000 | Train Loss: 0.013840 | Val Loss: 0.007792
Epoch 30/1000 | Train Loss: 0.012124 | Val Loss: 0.006931
Epoch 40/1000 | Train Loss: 0.010197 | Val Loss: 0.006384
Epoch 50/1000 | Train Loss: 0.008662 | Val Loss: 0.006434
Epoch 60/1000 | Train Loss: 0.010176 | Val Loss: 0.008411
Epoch 70/1000 | Train Loss: 0.013965 | Val Loss: 0.010174
Epoch 80/1000 | Train Loss: 0.008083 | Val Loss: 0.007338
Epoch 90/1000 | Train Loss: 0.007204 | Val Loss: 0.007282
Epoch 100/1000 | Train Loss: 0.006797 | Val Loss: 0.007448
Epoch 110/1000 | Train Loss: 0.006490 | Val Loss: 0.007511
Epoch 120/1000 | Train Loss: 0.006314 | Val Loss: 0.007519
Epoch 130/1000 | Train Loss: 0.006538 | Val Loss: 0.007551
Early stopping at epoch 137 due to no improvement in validation loss for 100 epochs.


In [ ]:
import importlib
import graph2vecinference_adaptativethreshold
importlib.reload(graph2vecinference_adaptativethreshold)
from graph2vecinference_adaptativethreshold import graph2vec_inference
import numpy as np

# Test Model Inference
item_id = products[0][0]
store_id = products[0][1]
# read the model from the best_model_path
model.load_state_dict(torch.load(best_model_path))
forecast, inference_time = graph2vec_inference(
    metric=metric,
    window_size=window_size,
    step_size=step_size,
    threshold=threshold,
    percentile=percentile,  # Added this 
    model=model,
    df=df,
    df_wide=None,
    cat_labels=None,
    date_col=DATE_COL,
    scaler=scaler,
    exog_scaler=exog_scaler,
    test_start_idx=test_start_idx,
    seq_length=seq_length,
    forecast_window=forecast_horizon,
    device=device,
    item_id=item_id,
    store_id=store_id,
    seed=seed,
    criterion="MSELoss",
    val_scaled=val_scaled,
    test_scaled=test_scaled, # <--- Added this parameter 
    exog_val_scaled=exog_val_scaled,
    exog_test_scaled=exog_test_scaled,
    exog_test_raw=exog_test,
    exog_cols=exog_cols,
    save_plot_path=None,
    node_embeddings=aligned_embeddings if USE_EMBEDDINGS else None,
    graph2vec_model=graph2vec_model if USE_EMBEDDINGS else None,
    enable_edges_within_star=enable_edges_within_star
)
if USE_EMBEDDINGS:
    prefix_star = "" if enable_edges_within_star else "star_"
    prefix = f"metric_{prefix_star}{metric}_res_{MODEL_TYPE}" if USE_RESIDUALS else f"metric_{prefix_star}{metric}"
    save_plot_path = f"grid_search_plots/item_{item_id}_store_{store_id}_{prefix}_window_{window_size}_step_{step_size}_threshold_{threshold}_percentile_{percentile}.png"
else:
    save_plot_path = None
    print("Skipping standalone baseline plot; baseline should be included in comparison plots.")
# Filter out the NaN values from the warmup period for metrics computation
valid_mask = ~np.isnan(forecast)
valid_test = test[valid_mask]
valid_forecast = np.array(forecast)[valid_mask]

try:
    from model_utils.plots import compute_metrics
    rmse, mae, bias, score , pocid = compute_metrics(valid_test, valid_forecast)
except ImportError:
    # If compute_metrics is missing, plots.py will compute them itself, but we need to ensure it uses the valid values
    rmse, mae, bias, score, pocid = None, None, None, None, None
    if len(valid_test) > 0:
        from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
        rmse = np.sqrt(mean_squared_error(valid_test, valid_forecast))
        mae = mean_absolute_error(valid_test, valid_forecast)
        bias = np.mean(valid_forecast - valid_test)
        score = r2_score(valid_test, valid_forecast)

if save_plot_path:
    train_index = df[DATE_COL][train_slice].values
    val_index = df[DATE_COL][val_slice].values
    test_index = df[DATE_COL][test_slice].values
    
    emb_title = f'with Graph2Vec Embeddings (Residuals: {USE_RESIDUALS})' if USE_EMBEDDINGS else 'Without Embeddings'
    plot_results(train, val, test, forecast, train_index, val_index, test_index,
                 train_losses, val_losses, metric=metric, embedding_strategy='graph2vec',
                 window_size=window_size, step_size=step_size, threshold=threshold, percentile=percentile,
                 enable_edges_within_star=enable_edges_within_star,
                 target_col=TARGET_COL, 
                 title=f'LSTM Forecast {emb_title} (Seed={seed}, Criterion=MSELoss, Item={item_id}, Store={store_id})',
                 save_path=save_plot_path,
                 rmse=rmse, mae=mae, bias=bias, score=score, pocid=pocid)

Test set starts on: 2022-09-24
Step 0: Predicting for Date: 2022-09-24
Step 10: Predicting for Date: 2022-10-04
Step 20: Predicting for Date: 2022-10-14
Step 30: Predicting for Date: 2022-10-24
Step 40: Predicting for Date: 2022-11-03
Step 50: Predicting for Date: 2022-11-13
Step 60: Predicting for Date: 2022-11-23
Step 70: Predicting for Date: 2022-12-03
Step 80: Predicting for Date: 2022-12-13
Step 90: Predicting for Date: 2022-12-23
Step 100: Predicting for Date: 2023-01-02
Step 110: Predicting for Date: 2023-01-12
Step 120: Predicting for Date: 2023-01-22
Step 130: Predicting for Date: 2023-02-01
Step 140: Predicting for Date: 2023-02-11
Step 150: Predicting for Date: 2023-02-21
Forecasted values [nan nan nan nan nan] ...
